# 4.6 缩放定律与涌现 / Scaling laws and emergence

模型变大、数据变多时损失怎样下降，固定算力下两者应当怎样分配，以及“涌现”这个说法为什么受到质疑。

How the loss falls as models and data grow, how a fixed compute budget should be split between the two, and why the notion of emergence has been questioned.

讲解与推导见网站对应模块。每个练习先写代码，再运行紧随其后的检查单元。

The explanation and derivations are on the website. For each exercise, write the code, then run the check cell right after it.

## 1. 幂律拟合与 Chinchilla 配比 / Power-law fitting and the Chinchilla allocation

实现双对数坐标下的幂律拟合、Chinchilla 论文中的参数化损失，以及固定算力下最优的参数量和 token 数。测试会用带噪声的数据检验拟合，并通过扫描其他分配方式确认闭式解确实最优。运行后请查看打印出的三个算力档位下的最优配置。

Implement power-law fitting in log-log coordinates, the parametric loss of the Chinchilla paper and the optimal numbers of parameters and tokens under a fixed compute budget. The tests check the fit on noisy data and confirm by scanning other allocations that the closed form is indeed optimal. Look at the printed optimal configurations for three compute budgets.

In [ ]:
import math


def fit_power_law(xs, ys):
    """Fit y = c * x^p by least squares in log-log space. Returns (c, p)."""
    # TODO 1: linear regression of log y on log x; the slope is p, exp(intercept) is c
    raise NotImplementedError


def chinchilla_loss(N, D, E=1.69, A=406.4, B=410.7, alpha=0.34, beta=0.28):
    """The parametric loss fitted by Hoffmann et al. (2022):  L(N, D) = E + A / N^alpha + B / D^beta

    N: number of parameters, D: number of training tokens.
    """
    # TODO 2
    raise NotImplementedError


def flops(N, D):
    """Training compute in floating-point operations, with the usual approximation C = 6 N D."""
    # TODO 3
    raise NotImplementedError


def optimal_allocation(C, A=406.4, B=410.7, alpha=0.34, beta=0.28):
    """The (N, D) that minimises the loss for a compute budget C = 6 N D.

    N_opt = G * (C / 6)^(beta / (alpha + beta))
    D_opt = (C / 6)^(alpha / (alpha + beta)) / G
    G     = (alpha * A / (beta * B))^(1 / (alpha + beta))
    Returns (N_opt, D_opt) as floats.
    """
    # TODO 4
    raise NotImplementedError

In [ ]:
# 检查 / checks
import math, random

c, p = fit_power_law([1e6, 1e7, 1e8, 1e9], [5.0 * x ** -0.076 for x in (1e6, 1e7, 1e8, 1e9)])
assert abs(c - 5.0) < 1e-6 and abs(p + 0.076) < 1e-9
rng = random.Random(0)
xs = [10 ** (6 + 0.25 * i) for i in range(17)]
ys = [3.0 * x ** -0.1 * math.exp(rng.gauss(0, 0.02)) for x in xs]
c, p = fit_power_law(xs, ys)
assert abs(p + 0.1) < 0.01, f"with 2% noise the exponent should still be recovered, got {p:.3f}"

assert abs(chinchilla_loss(1e9, 1e11) - (1.69 + 406.4 / 1e9 ** 0.34 + 410.7 / 1e11 ** 0.28)) < 1e-12
assert chinchilla_loss(1e12, 1e15) > 1.69, "the loss never goes below the irreducible term E"
assert chinchilla_loss(2e9, 1e11) < chinchilla_loss(1e9, 1e11) and chinchilla_loss(1e9, 2e11) < chinchilla_loss(1e9, 1e11)
assert flops(1e9, 1e11) == 6e20

for C in (1e19, 1e21, 1e23):
    N, D = optimal_allocation(C)
    assert abs(flops(N, D) / C - 1) < 1e-9, "the allocation must use exactly the budget"
    best = chinchilla_loss(N, D)
    for k in range(-40, 41):                       # scan other ways of splitting the same budget
        N2 = N * 10 ** (k / 20); D2 = C / (6 * N2)
        assert chinchilla_loss(N2, D2) >= best - 1e-12, "another split of the same budget does better"

# how the optimum moves with the budget
(N1, D1), (N2, D2) = optimal_allocation(1e20), optimal_allocation(1e22)
a = math.log(N2 / N1) / math.log(100); b = math.log(D2 / D1) / math.log(100)
assert abs(a - 0.28 / 0.62) < 1e-9 and abs(b - 0.34 / 0.62) < 1e-9
assert abs(a - 0.45) < 0.01 and abs(b - 0.55) < 0.01, "parameters and tokens grow at nearly the same rate"
print("m46_scaling: all tests passed")
for C in (1e19, 1e21, 1e23):
    N, D = optimal_allocation(C)
    print(f"  C = {C:.0e} FLOPs: N = {N:.2e} parameters, D = {D:.2e} tokens, loss {chinchilla_loss(N, D):.3f}")

<details><summary><b>提示 1 / Hint 1</b></summary>

幂律 $y = c\,x^p$ 取对数后是 $\log y = \log c + p \log x$，对 $\log x$ 和 $\log y$ 做一元线性回归，斜率是 $p$，截距取指数是 $c$。

Taking logs of the power law $y = c\,x^p$ gives $\log y = \log c + p \log x$. A simple linear regression of $\log y$ on $\log x$ has slope $p$, and the exponential of the intercept is $c$.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

最优配比照函数说明中的三个公式写即可，先算 $G$。

Write the optimal allocation from the three formulas of the docstring, computing $G$ first.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import math


def fit_power_law(xs, ys):
    u = [math.log(x) for x in xs]
    v = [math.log(y) for y in ys]
    mu, mv = sum(u) / len(u), sum(v) / len(v)
    p = sum((a - mu) * (b - mv) for a, b in zip(u, v)) / sum((a - mu) ** 2 for a in u)
    return math.exp(mv - p * mu), p


def chinchilla_loss(N, D, E=1.69, A=406.4, B=410.7, alpha=0.34, beta=0.28):
    return E + A / N ** alpha + B / D ** beta


def flops(N, D):
    return 6 * N * D


def optimal_allocation(C, A=406.4, B=410.7, alpha=0.34, beta=0.28):
    G = (alpha * A / (beta * B)) ** (1 / (alpha + beta))
    N = G * (C / 6) ** (beta / (alpha + beta))
    D = (C / 6) ** (alpha / (alpha + beta)) / G
    return N, D
```

</details>

## 2. 指标怎样让平滑的进步看起来像突变 / How a metric makes smooth progress look like a sudden change

给定一个随模型规模平稳上升的逐 token 正确率，计算不同答案长度下的完全匹配率，并度量曲线上最大的跳变。测试会检查，在 30 个 token 的完全匹配指标下，小模型的得分几乎为 0，最大的模型接近 1，跳变幅度是逐 token 正确率的 5 倍以上。这里的正确率规律是为演示而设定的，不是任何真实模型的数据。

Given a per-token accuracy that rises steadily with model size, compute the exact-match rate for different answer lengths and measure the largest jump on the curve. The tests check that under exact match over 30 tokens the small models score almost 0, the largest scores close to 1 and the jump is more than five times that of the per-token accuracy. The accuracy law used here is made up for demonstration and is not data from any real model.

In [ ]:
import math


def per_token_accuracy(n_params, slope=0.1165, at_1e6=0.30):
    """A made-up but smooth law: accuracy on a single token grows linearly in log10(parameters).

    Capped at 0.999. Nothing to change here.
    """
    return min(0.999, at_1e6 + slope * (math.log10(n_params) - 6))


def exact_match(p, length):
    """Probability of getting ALL `length` tokens of an answer right, if each is right with probability p (independently)."""
    # TODO 1
    raise NotImplementedError


def biggest_jump(values):
    """The largest increase between two neighbouring entries of a list."""
    # TODO 2
    raise NotImplementedError


def threshold_scale(sizes, scores, level=0.5):
    """The first model size whose score reaches `level`, or None if none does."""
    # TODO 3
    raise NotImplementedError

In [ ]:
# 检查 / checks
import math

assert abs(exact_match(0.9, 1) - 0.9) < 1e-12 and abs(exact_match(0.9, 10) - 0.9 ** 10) < 1e-12
assert biggest_jump([0.1, 0.2, 0.25, 0.7, 0.8]) == 0.7 - 0.25
assert threshold_scale([1, 10, 100], [0.2, 0.6, 0.9]) == 10 and threshold_scale([1, 10], [0.1, 0.2]) is None

sizes = [10 ** (6 + 0.5 * i) for i in range(13)]                # 1e6 ... 1e12 parameters
p = [per_token_accuracy(n) for n in sizes]
assert biggest_jump(p) < 0.06, "per-token accuracy improves in small, even steps"

em1 = [exact_match(q, 1) for q in p]
em30 = [exact_match(q, 30) for q in p]
assert em1 == p
# The same models, scored on a 30-token answer with exact match:
assert max(em30[:8]) < 0.02, "small models look as if they had no ability at all"
assert em30[-1] > 0.9, "the largest model nearly always succeeds"
assert biggest_jump(em30) > 5 * biggest_jump(p), "the curve has a sudden jump although nothing sudden happened underneath"
assert threshold_scale(sizes, em30) > 100 * threshold_scale(sizes, em1), "the ability seems to 'appear' at a much larger scale"

# A continuous metric on the same task shows the smooth trend again: expected number of correct tokens.
expected_correct = [30 * q for q in p]
assert biggest_jump(expected_correct) / 30 < 0.06
print("m46_emergence: all tests passed")
print("  log10(params):", [round(math.log10(n), 1) for n in sizes])
print("  per-token acc:", [round(q, 2) for q in p])
print("  exact match, 30 tokens:", [round(q, 3) for q in em30])

<details><summary><b>提示 1 / Hint 1</b></summary>

各 token 独立时，全部正确的概率是 `p ** length`。

With independent tokens the probability of all being correct is `p ** length`.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

最大跳变用 `max(b - a for a, b in zip(values, values[1:]))`。

The largest jump is `max(b - a for a, b in zip(values, values[1:]))`.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import math


def per_token_accuracy(n_params, slope=0.1165, at_1e6=0.30):
    return min(0.999, at_1e6 + slope * (math.log10(n_params) - 6))


def exact_match(p, length):
    return p ** length


def biggest_jump(values):
    return max(b - a for a, b in zip(values, values[1:]))


def threshold_scale(sizes, scores, level=0.5):
    for n, s in zip(sizes, scores):
        if s >= level:
            return n
    return None
```

</details>

## 3. 推导最优参数与数据配比 / Derive the optimal split between parameters and data

合上上面的推导，在纸上完成。
(a) 在约束 $6ND = C$ 下最小化 $A N^{-\alpha} + B D^{-\beta}$，推导 $N_{\text{opt}}$ 和 $D_{\text{opt}}$。
(b) 代入 $\alpha = 0.34$、$\beta = 0.28$，$N_{\text{opt}}$ 和 $D_{\text{opt}}$ 分别随 $C$ 的多少次方增长？
(c) 算力增加到 100 倍，最优的参数量和 token 数各变为原来的多少倍？
(d) 一个需要连续答对 30 个 token 的任务。逐 token 正确率从 0.90 提高到 0.95，完全匹配率从多少变到多少？这说明了什么？

Close the derivation above and work on paper.
(a) Minimize $A N^{-\alpha} + B D^{-\beta}$ under the constraint $6ND = C$ and derive $N_{\text{opt}}$ and $D_{\text{opt}}$.
(b) With $\alpha = 0.34$ and $\beta = 0.28$, with what powers of $C$ do $N_{\text{opt}}$ and $D_{\text{opt}}$ grow?
(c) Compute increases a hundredfold. By what factors do the optimal numbers of parameters and tokens change?
(d) A task requires 30 consecutive correct tokens. Per-token accuracy rises from 0.90 to 0.95. From what to what does exact match change, and what does this show?

<details><summary><b>提示 1 / Hint 1</b></summary>

(a) 用 $D = C/(6N)$ 消去 $D$，对 $N$ 求导并令其为 0。

(a) Eliminate $D$ with $D = C/(6N)$, differentiate with respect to $N$ and set to 0.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

(c) 计算 $100^{0.45}$ 和 $100^{0.55}$。

(c) Compute $100^{0.45}$ and $100^{0.55}$.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

(a) 代入 $D = C/(6N)$ 得 $f(N) = A N^{-\alpha} + B (C/6)^{-\beta} N^{\beta}$。令 $f'(N) = -\alpha A N^{-\alpha-1} + \beta B (C/6)^{-\beta} N^{\beta - 1} = 0$，得 $N^{\alpha+\beta} = \dfrac{\alpha A}{\beta B} (C/6)^{\beta}$。所以 $N_{\text{opt}} = G\,(C/6)^{\beta/(\alpha+\beta)}$，$D_{\text{opt}} = G^{-1} (C/6)^{\alpha/(\alpha+\beta)}$，其中 $G = \big(\tfrac{\alpha A}{\beta B}\big)^{1/(\alpha+\beta)}$。

(b) $N_{\text{opt}} \propto C^{0.28/0.62} \approx C^{0.45}$，$D_{\text{opt}} \propto C^{0.34/0.62} \approx C^{0.55}$。

(c) 参数量变为 $100^{0.45} \approx 8$ 倍，token 数变为 $100^{0.55} \approx 12.5$ 倍，两者的乘积是 100。

(d) $0.90^{30} \approx 0.042$，$0.95^{30} \approx 0.215$。逐 token 正确率只提高了约 6%，完全匹配率却变成了原来的 5 倍。同一个进步在不同的指标下看起来差别很大，非线性的指标会把平滑的提升显示为陡峭的跳变。

---

(a) Substituting $D = C/(6N)$ gives $f(N) = A N^{-\alpha} + B (C/6)^{-\beta} N^{\beta}$. Setting $f'(N) = -\alpha A N^{-\alpha-1} + \beta B (C/6)^{-\beta} N^{\beta - 1} = 0$ gives $N^{\alpha+\beta} = \dfrac{\alpha A}{\beta B} (C/6)^{\beta}$. Hence $N_{\text{opt}} = G\,(C/6)^{\beta/(\alpha+\beta)}$ and $D_{\text{opt}} = G^{-1} (C/6)^{\alpha/(\alpha+\beta)}$ with $G = \big(\tfrac{\alpha A}{\beta B}\big)^{1/(\alpha+\beta)}$.

(b) $N_{\text{opt}} \propto C^{0.28/0.62} \approx C^{0.45}$ and $D_{\text{opt}} \propto C^{0.34/0.62} \approx C^{0.55}$.

(c) Parameters grow by $100^{0.45} \approx 8$ and tokens by $100^{0.55} \approx 12.5$, and the product of the two is 100.

(d) $0.90^{30} \approx 0.042$ and $0.95^{30} \approx 0.215$. Per-token accuracy rises by only about 6% while exact match becomes five times as large. The same progress looks very different under different metrics, and a nonlinear metric displays a smooth gain as a steep jump.

</details>